# バンディット問題勉強用ノートブック
このノートブックは、1状態のマルコフ決定過程（MDP）と形式的に等価なバンディット問題を説明するために作成されました。バンディット問題は、有限のアクション集合 $\mathcal{A}$ と、現在のアクションに対する報酬の確率分布 $p$ のタプル $(\mathcal{A}, p)$ で定義されます。以下の import 文は必要なモジュールやパッケージを読み込むためのもので、最初は意味が分からなくても大丈夫です。必要ならば Python の本を参照してください。

In [ ]:
# rlutils パッケージのインストール（初回のみ）
# Colab の場合は以下を実行してください
# !pip install -q git+https://github.com/uchibe/CPAI_study.git
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, fixed
%matplotlib inline

# 共通ユーティリティを rlutils からインポート
from rlutils import (
    GaussianBandit,
    argmax_random_tie,
    epsilon_greedy_policy,
    naive_epsilon_greedy_policy,
    boltzmann_policy,
    select_ucb_action,
    uniform_random_method,
    value_based_method,
    ucb_method,
    run_experiment,
    plot_epsilon_greedy_policy,
    plot_boltzmann_policy,
    compare_policies,
)

## バンディット問題の例
5台のスロットマシンがあり、エージェントは5つの離散的な行動を選択できます。エージェントはアクションの選択に応じてスカラー報酬を受け取ります。目標は、期待報酬を最大化する行動を見つけることです。

**注意点**

- `GaussianBandit` は教育用のシンプルなクラスです（`gymnasium.Env` のサブクラスではありません）。
- すべての確率的なルーチンは `seed` 引数を受け付け、複数シードの平均をとるための `run_experiment` を提供しています。

## 報酬を与える確率分布の可視化
この例題ではガウス分布で報酬を確率的に生成しています。
先頭の`seed=2026`となっている箇所の数値 2026 を学籍番号の数値と置き換えてからセルを実行してください。

レポートには`env.report_settings()`で表示される5個のガウス分布の平均と標準偏差を明記してください。

In [ ]:
env = GaussianBandit(seed=2026)
env.report_settings()
env.plot()

## 一様ランダム方策

より洗練されたアクション選択手法を検討する前に、シンプルな一様ランダム方策でベースラインを確認しましょう。

In [ ]:
_ = uniform_random_method(env=env, number_of_steps=1000)

## $\varepsilon$-greedy方策
$\varepsilon$-greedyは、価値ベースの強化学習において探索（exploration）と活用（exploitation）のバランスをとるための手法です。現在の行動価値関数の推定値が $Q(a)$ であるとき、行動 $a$ を選択する確率は以下のように与えられます。
\begin{equation}
  \pi (a) =
  \begin{cases}
    \epsilon/\lvert A \rvert + (1 - \epsilon)/|\mathrm{argmax}_a \,Q| & \mathrm{if} \; \;
       a \in \mathrm{arg}\max_a Q(a), \\
    \epsilon/\lvert A \rvert & \mathrm{otherwise},
  \end{cases}
\end{equation}
`epsilon_greedy_policy` は `rlutils.policies` で定義されています。

In [ ]:
Q = np.array([2.5, -2.5, 2.9, 1.2, 0.5])
interact(plot_epsilon_greedy_policy, Q=fixed(Q), epsilon=(0, 1, 0.05));

### タイブレークの影響の可視化
すべての $Q$ 値が $0$ である初期状態において、両者の挙動の違いを確認します。

In [ ]:
Q_initial = np.zeros(5)
print("Initial Q values:", Q_initial)
compare_policies(Q_initial, epsilon=0.1)

# 価値ベースの手法

エージェントは期待報酬を学習します。エージェントが行動 $a$ を選択し、スカラー報酬 $r$ を受け取ると、期待報酬の推定値は次のように更新されます。
\begin{equation*}
  Q(a) = Q(a) + \alpha (r - Q(a)),
\end{equation*}
ここで $\alpha$ は学習率です。`value_based_method` は `rlutils.methods` で定義されています。

In [ ]:
current_run_seed = np.random.randint(0, 1_000_000)
epsilon = 0.05
_ = value_based_method(env=env, epsilon=epsilon, seed=current_run_seed)

## UCB (Upper Confidence Bound) 方策
$$ \text{UCB}_t(a) = Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} $$
`select_ucb_action` と `ucb_method` は `rlutils` で定義されています。

In [ ]:
current_run_seed = np.random.randint(0, 1_000_000)
_ = ucb_method(env=env, c=2.0, seed=current_run_seed)

## 複数回の実行による平均化と比較

1回だけの実行（シングルラン）はノイズが多いため、多くの独立したシードで平均をとります。

In [ ]:
n_runs = 200
T = 1000

configs = [
    dict(label=r'$\varepsilon$=0.1, $Q_0$=0',  method_function=value_based_method, epsilon=0.1,  initial_Q=0.0),
    dict(label=r'$\varepsilon$=0.01, $Q_0$=0', method_function=value_based_method, epsilon=0.01, initial_Q=0.0),
    dict(label=r'greedy, $Q_0$=10 (optimistic)', method_function=value_based_method, epsilon=0.0,  initial_Q=10.0),
    dict(label=r'UCB c=2', method_function=ucb_method, c=2.0, initial_Q=0.0),
]

results = []
for cfg in configs:
    current_cfg = cfg.copy()
    label = current_cfg.pop('label')
    method_func = current_cfg.pop('method_function')
    avg_r, opt_r, Q_last = run_experiment(env=env, method_function=method_func,
                                          n_runs=n_runs, number_of_steps=T, **current_cfg)
    results.append((label, avg_r, opt_r, Q_last))

fig, axarr = plt.subplots(1, 2, figsize=(13, 5))
for label, avg_r, opt_r, _ in results:
    axarr[0].plot(avg_r, label=label)
    axarr[1].plot(opt_r, label=label)
axarr[0].set_xlabel('steps'); axarr[0].set_ylabel('average reward')
axarr[0].grid(); axarr[0].legend()
axarr[1].set_xlabel('steps'); axarr[1].set_ylabel('optimal action rate')
axarr[1].grid(); axarr[1].legend()
plt.tight_layout(); plt.show()